In [4]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AgentEndpointConfig,
    FixedRatioVersionSelectionRule,
    VersionSelector,
)
from azure.identity import DefaultAzureCredential

In [5]:
PROJECT_ENDPOINT = "https://foundry-rag-nagh.services.ai.azure.com/api/projects/rag"

agent_name = "agent-rag"

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

with project_client:
    endpoint_config = AgentEndpointConfig(
        version_selector=VersionSelector(
            version_selection_rules=[
                FixedRatioVersionSelectionRule(agent_version="4", traffic_percentage=100),
            ]
        ),
    )

    patched_agent = project_client.agents.update_details(
        agent_name=agent_name,
        agent_endpoint=endpoint_config,
    )
    print(f"Agent endpoint configured for agent: {patched_agent.name}")

Agent endpoint configured for agent: agent-rag


In [10]:
openai = project_client.get_openai_client()

# Create conversation and send request with runtime values
conversation = openai.conversations.create()
response = openai.responses.create(
    conversation=conversation.id,
    input="Hello! Can you confirm my details?",
    extra_body={
        "agent_reference": {"name": patched_agent.name, "type": "agent_reference"},
        "structured_inputs": {"userName": "Alice Smith", "userRole": "Senior Developer"},
    },
)
print(response.output_text)

BadRequestError: Error code: 400 - {'error': {'code': 'invalid_payload', 'message': 'Unknown inputs [userName, userRole]. No such structured inputs are defined on the agent. [Request ID: bcf2493589f61c74da960ba046d245a1]', 'param': 'structured_inputs', 'type': 'invalid_request_error', 'details': [{'code': 'ValidationError', 'message': 'Unknown inputs [userName, userRole]. No such structured inputs are defined on the agent.', 'param': 'structured_inputs', 'type': 'error', 'details': []}], 'additionalInfo': {'request_id': 'bcf2493589f61c74da960ba046d245a1'}}}